# San Francisco Can't Agree on North

Streets closed into city blocks, each colored by the way it points. Uses the Day 3 streets from `fetch_week1.py day3`.

## 1 · Settings

In [ ]:
DAY      = 3
PROMPT   = "Polygons"
SLUG     = "polygons-sf-grids"
NOTEBOOK = "day03_polygons_sf_grids.ipynb"
TITLE    = "San Francisco\nCan't Agree on North"
CREDITS  = "Streets: © OpenStreetMap contributors. City boundary: US Census Bureau."

## 2 · Load

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, pandas as pd, shapely
from matplotlib.colors import LinearSegmentedColormap
from mapkit import poster, publish
from mapkit.paths import PROCESSED

streets = gpd.read_file(PROCESSED / "sf_streets.geojson")
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'sf'")

## 3 · Analysis, every number on the map comes from here

In [ ]:
UTM = 32610
lines = streets.to_crs(UTM).geometry
land = city.to_crs(UTM).geometry.iloc[0]

# Close the street lines into polygons: each one is a city block
blocks = gpd.GeoDataFrame(geometry=list(shapely.polygonize([shapely.union_all(lines.values)]).geoms), crs=UTM)
blocks["m2"] = blocks.area
blocks = blocks[blocks.representative_point().within(land)]
blocks = blocks[blocks.m2.between(1200, 450000)].copy()      # drop slivers, and parks that aren't really blocks

def direction(g):
    """Which way a block points, in degrees east of north. Grids repeat every 90 degrees, so 0 to 90."""
    r = np.array(g.minimum_rotated_rectangle.exterior.coords)
    a, b = r[1] - r[0], r[2] - r[1]
    long_side = a if np.hypot(*a) >= np.hypot(*b) else b
    return np.degrees(np.arctan2(long_side[0], long_side[1])) % 90

blocks["deg"] = blocks.geometry.map(direction)
hist, edges = np.histogram(blocks.deg, bins=18, range=(0, 90), weights=blocks.m2)
print(f"Blocks drawn  {len(blocks):,}")
for count, left in sorted(zip(hist, edges), reverse=True)[:6]:
    print(f"  {left:4.0f} to {left + 5:.0f} deg   {count / blocks.m2.sum() * 100:4.1f}% of block area")

def typical(lon, lat, km=0.6):
    """The most common direction near a point."""
    p = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(UTM).iloc[0]
    near = blocks[blocks.distance(p) < km * 1000]
    h, e = np.histogram(near.deg, bins=18, range=(0, 90), weights=near.m2)
    return float(e[h.argmax()] + 2.5), len(near)

for name, lon, lat in [("Sunset", -122.485, 37.755), ("Richmond", -122.48, 37.78), ("Financial District", -122.405, 37.793),
                       ("SoMa", -122.402, 37.78), ("Mission", -122.417, 37.76), ("Outer Mission", -122.44, 37.72)]:
    print(f"  {name:20} {typical(lon, lat)}")

## 4 · Map

In [ ]:
PAPER = "#fbf6ea"
ramp = LinearSegmentedColormap.from_list("grids", ["#1d4e9e", "#0f9b8e", "#f4b41a", "#e8402c", "#a3238e", "#1d4e9e"])
fig, ax = poster.page(PAPER)
m = blocks.to_crs(3857)
m.plot(ax=ax, color=[ramp(d / 90) for d in m.deg], edgecolor=PAPER, lw=0.25, zorder=2)
ax.set_axis_off()
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.5165, 37.704, -122.355, 37.8125)
h = fy1 - fy0
ax.set_ylim(fy0 + h * 0.035, fy1 + h * 0.035)

INK = "#17202a"
fig.text(0.06, 0.968, "San Francisco", color=INK, fontproperties=poster.font(poster.ABRIL, 50), va="top")
fig.text(0.06, 0.91, "Can't Agree on North", color=INK, fontproperties=poster.font(poster.ABRIL, 50), va="top")
fig.text(0.06, 0.845, "Every city block, colored by the way it points. South of Market Street,\nthe grid swings about 45 degrees off the blocks just across the road.", color="#2b3440",
         fontproperties=poster.font(poster.POPPINS, 14), va="top", linespacing=1.4)

def label(lon, lat, text, ha="center"):
    x, y = poster.merc(lon, lat)
    poster.halo_text(ax, x, y, text, color=INK, path=poster.POPPINS_BOLD, size=12.5, halo=PAPER, halo_width=3.2, ha=ha, va="center", zorder=9, linespacing=1.2)

label(-122.402, 37.780, "South of\nMarket")
label(-122.487, 37.752, "The Sunset")
label(-122.443, 37.752, "Twin Peaks:\nno grid at all")
label(-122.433, 37.783, "North of Market")

# a compass ring as the legend: the colors repeat every 90 degrees, like a grid does
ring = fig.add_axes([0.05, 0.135, 0.125, 0.1], projection="polar")
ring.set_theta_zero_location("N"); ring.set_theta_direction(-1)
th, rr = np.meshgrid(np.linspace(0, 2 * np.pi, 361), [1.5, 2.5])
ring.pcolormesh(th, rr, (np.degrees(th[:-1, :-1]) % 90) / 90, cmap=ramp, vmin=0, vmax=1, shading="flat")
ring.set_ylim(0, 2.5); ring.set_axis_off()
fig.text(0.1125, 0.242, "N", ha="center", va="bottom", color=INK, fontproperties=poster.font(poster.POPPINS_BOLD, 10.5))
poster.credit(fig, CREDITS, path=poster.POPPINS, name_path=poster.POPPINS_BOLD, color="#55606b", name_color="#17202a", size=9.5)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle="Every city block, colored by the way it points", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)